# 🧠 Mental Health Google Form — Research Analysis
### Ensemble: Your Model (LR/SVM/RF) + BART Zero-Shot
**Inputs:** Google Form CSV  |  **Models:** Reddit TF-IDF ensemble + facebook/bart-large-mnli  |  **XAI:** SHAP

---
> **Run cells in order. Only edit the `⚙️ CONFIGURATION` cell.**

In [ ]:
from pathlib import Path as _ProjectPath
import sys as _project_sys

_project_root = next(
    (
        p
        for p in (_ProjectPath.cwd(), *_ProjectPath.cwd().parents)
        if (p / "project_config.json").is_file()
    ),
    None,
)
if _project_root is None:
    raise FileNotFoundError(
        "Open VS Code in the organized project root before running this notebook."
    )
if str(_project_root / "tools") not in _project_sys.path:
    _project_sys.path.insert(0, str(_project_root / "tools"))
from project_paths import ProjectPaths as _ProjectPaths

_paths = _ProjectPaths(_project_root, 'google_form_part2')

import os, warnings, re

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import shap
import chardet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

plt.rcParams.update(
    {
        'figure.dpi': 120,
        'figure.facecolor': 'white',
        'axes.facecolor': '#f9f9f9',
        'axes.grid': True,
        'grid.alpha': 0.3,
        'font.family': 'DejaVu Sans',
    }
)
PALETTE = [
    '#e74c3c',
    '#3498db',
    '#2ecc71',
    '#f39c12',
    '#9b59b6',
    '#1abc9c',
    '#e67e22',
    '#34495e',
    '#e91e63',
    '#00bcd4',
    '#8bc34a',
    '#ff5722',
    '#607d8b',
    '#ffc107',
    '#795548',
]

print("✅ Libraries loaded")
print(f"   SHAP version : {shap.__version__}")
print(f"   Pandas       : {pd.__version__}")
print(f"   NumPy        : {np.__version__}")


In [ ]:
CSV_FILE = _paths.input_path('Mentalhealth.csv')
Q1_KEYWORD = "heaviest"
Q2_KEYWORD = "alone"
Q1_COL_INDEX = None
Q2_COL_INDEX = None
COMBINE_RESPONSES = True
CONFIDENCE_THRESHOLD = 0.38
BORDERLINE_LOWER = 0.28
MODEL_DIR = _paths.classical_dir
TFIDF_FILE = "tfidf_vectorizer.pkl"
LABEL_FILE = "label_encoder.pkl"
LR_FILE = "logistic_regression_model.pkl"
SVM_FILE = "linear_svm_model.pkl"
RF_FILE = "random_forest_model.pkl"

print("✅ Configuration set")
print(f"   CSV          : {CSV_FILE}")
print(f"   Threshold    : {CONFIDENCE_THRESHOLD}")


In [ ]:
print("🤖 Loading pre-trained models...")
print("=" * 70)

import os

MODEL_DIR = _paths.classical_dir
_test_path = os.path.join(MODEL_DIR, TFIDF_FILE)
if not os.path.exists(_test_path):
    MODEL_DIR = _paths.classical_dir

print(f"   📂 Loading from: {MODEL_DIR}")


def _load(filename):
    path = os.path.join(MODEL_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"\n❌ Cannot find: {path}")
    return joblib.load(_paths.input_path(path))


tfidf = _load(TFIDF_FILE)
print(f"   ✅ TF-IDF vectorizer  ({tfidf.max_features} features)")
le = _load(LABEL_FILE)
print(f"   ✅ Label encoder      ({len(le.classes_)} classes)")
lr = _load(LR_FILE)
print(f"   ✅ Logistic Regression")
svm = _load(SVM_FILE)
print(f"   ✅ Linear SVM")
rf = _load(RF_FILE)
print(f"   ✅ Random Forest")

models = {"Logistic Regression": lr, "Linear SVM": svm, "Random Forest": rf}
print(f"\n🎯 Ready — {len(le.classes_)} mental-health categories available")
print(f"   Classes: {list(le.classes_)}")


In [ ]:
print(f"📂 Loading: {CSV_FILE}")
print("=" * 70)

csv_path = _paths.input_path(
    CSV_FILE if os.path.isabs(CSV_FILE) else os.path.join(MODEL_DIR, CSV_FILE)
)

raw_df = None
for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
    try:
        raw_df = pd.read_csv(
            _paths.input_path(csv_path),
            encoding=enc,
            on_bad_lines="skip",
            engine="python",
        )
        print(f"   ✅ Encoding '{enc}' worked")
        break
    except UnicodeDecodeError:
        print(f"   ⚠️  '{enc}' failed, trying next...")

if raw_df is None:
    raise ValueError("❌ Could not read CSV.")

print(f"\n✅ Loaded  →  {raw_df.shape[0]} rows × {raw_df.shape[1]} columns")
for i, col in enumerate(raw_df.columns):
    print(f"   [{i}] {col[:70]}")
raw_df.head(3)


In [ ]:
def _find_col(df, keyword=None, index=None, label="Q"):
    if index is not None:
        col = df.columns[index]
        print(f"   ✅ {label} → column [{index}]: {col[:70]}")
        return col
    if keyword:
        matches = [c for c in df.columns if keyword.lower() in c.lower()]
        if matches:
            print(f"   ✅ {label} → keyword '{keyword}' matched: {matches[0][:70]}")
            return matches[0]
    raise ValueError(f"Could not find {label} column.")


print("🔍 Detecting response columns...")
q1_col = _find_col(raw_df, Q1_KEYWORD, Q1_COL_INDEX, "Q1")
q2_col = _find_col(raw_df, Q2_KEYWORD, Q2_COL_INDEX, "Q2")

df = pd.DataFrame()
df["q1"] = raw_df[q1_col].fillna("").astype(str).str.strip()
df["q2"] = raw_df[q2_col].fillna("").astype(str).str.strip()
df["participant_id"] = [f"P{i+1:03d}" for i in range(len(df))]

ts_col = [c for c in raw_df.columns if "timestamp" in c.lower()]
if ts_col:
    df["timestamp"] = pd.to_datetime(raw_df[ts_col[0]], errors="coerce")
    print(f"   ✅ Timestamp column detected")

if COMBINE_RESPONSES:
    df["text"] = (df["q1"] + " " + df["q2"]).str.strip()
    print(f"\n✅ Combined Q1+Q2 into single 'text' column")
else:
    df["text"] = df["q1"]

df = df[df["text"].str.len() > 10].reset_index(drop=True)
df["word_count"] = df["text"].apply(lambda x: len(x.split()))

print(f"\n📊 Final sample: {len(df)} valid participants")
print(
    f"   Word count: min={df['word_count'].min()}, max={df['word_count'].max()}, mean={df['word_count'].mean():.0f}"
)
df[["participant_id", "word_count", "text"]].head(4)


In [ ]:
from transformers import pipeline
import torch, torch.nn.functional as F

print("Loading zero-shot classifier (no login required)...")
zs_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1,
)
print("   Loaded: facebook/bart-large-mnli")


SCHEMA_LABELS = [
    'addiction',
    'adhd',
    'alcoholism',
    'anxiety',
    'autism',
    'bipolarreddit',
    'bpd',
    'depression',
    'healthanxiety',
    'lonely',
    'mentalhealth',
    'ptsd',
    'schizophrenia',
    'socialanxiety',
    'suicidewatch',
]

ZS_LABELS = [
    "normal and mentally healthy",
    "anxiety and chronic worry or stress",
    "depression and persistent low mood",
    "mood disorder with emotional highs and lows",
    "grief or trauma from loss or past events",
    "suicidal thoughts or crisis",
    "ADHD and attention or focus difficulties",
    "general mental health distress",
]
ZS_TO_SCHEMA = {
    "normal and mentally healthy": "normal",
    "anxiety and chronic worry or stress": "anxiety",
    "depression and persistent low mood": "depression",
    "mood disorder with emotional highs and lows": "mood_disorder",
    "grief or trauma from loss or past events": "trauma_grief",
    "suicidal thoughts or crisis": "crisis",
    "ADHD and attention or focus difficulties": "adhd",
    "general mental health distress": "distress",
}


SCHEMA_TO_REDDIT = {
    "depression": "depression",
    "anxiety": "anxiety",
    "crisis": "suicidewatch",
    "mood_disorder": "bipolarreddit",
    "trauma_grief": "ptsd",
    "adhd": "adhd",
    "distress": "mentalhealth",
    "normal": "normal",
}

REDDIT_MAP = {
    "depression": "depression",
    "suicidewatch": "crisis",
    "bpd": "mood_disorder",
    "bipolarreddit": "mood_disorder",
    "anxiety": "anxiety",
    "socialanxiety": "anxiety",
    "healthanxiety": "anxiety",
    "adhd": "adhd",
    "ptsd": "trauma_grief",
    "lonely": "depression",
    "mentalhealth": "distress",
    "schizophrenia": "distress",
    "autism": "distress",
    "alcoholism": "distress",
    "addiction": "distress",
    "normal": "normal",
}

REDDIT_FREQ = {
    "depression": 0.30,
    "suicidewatch": 0.12,
    "mentalhealth": 0.10,
    "anxiety": 0.07,
    "adhd": 0.06,
    "lonely": 0.06,
    "socialanxiety": 0.05,
    "bipolarreddit": 0.05,
    "bpd": 0.04,
    "ptsd": 0.04,
    "healthanxiety": 0.03,
    "autism": 0.03,
    "schizophrenia": 0.02,
    "alcoholism": 0.02,
    "addiction": 0.01,
    "normal": 0.01,
}
UNIFORM = 1.0 / len(le.classes_)
CORRECTION = np.array([UNIFORM / REDDIT_FREQ.get(c, UNIFORM) for c in le.classes_])

DISTRESS_WORDS = {
    "death",
    "die",
    "dying",
    "dead",
    "kill",
    "suicide",
    "suicidal",
    "worthless",
    "hopeless",
    "empty",
    "numb",
    "hurt",
    "harm",
    "useless",
    "meaningless",
    "depressed",
    "anxiety",
    "panic",
    "trauma",
    "abuse",
    "exhausted",
    "stuck",
    "lonely",
    "trapped",
    "cry",
    "crying",
    "dark",
}


def run_your_model(texts):
    X = tfidf.transform(texts)
    probas = []
    for model in models.values():
        if hasattr(model, "predict_proba"):
            p = model.predict_proba(X)
        else:
            d = model.decision_function(X)
            p = np.exp(d - d.max(axis=1, keepdims=True))
            p /= p.sum(axis=1, keepdims=True)
        probas.append(p)
    avg = np.stack(probas).mean(axis=0)
    avg = avg * CORRECTION
    avg = avg / avg.sum(axis=1, keepdims=True)
    return avg


def run_zeroshot(text):
    result = zs_classifier(text[:1024], candidate_labels=ZS_LABELS, multi_label=False)
    return {
        ZS_TO_SCHEMA[lbl]: float(score)
        for lbl, score in zip(result["labels"], result["scores"])
    }


def aggregate_to_schema(raw_probs_dict, label_map):
    schema = {l: 0.0 for l in SCHEMA_LABELS}
    for raw_label, prob in raw_probs_dict.items():
        mapped = label_map.get(raw_label.lower(), "distress")
        schema[mapped] += prob
    total = sum(schema.values()) or 1.0
    return {k: v / total for k, v in schema.items()}


def classify_batch(texts, threshold=CONFIDENCE_THRESHOLD):
    your_proba = run_your_model(texts)
    rows, blended_rows = [], []

    print(f"\n   Classifying {len(texts)} participants (Your model + BART ensemble)...")
    for i, text in enumerate(texts):
        your_dict = {
            le.classes_[j]: float(your_proba[i][j]) for j in range(len(le.classes_))
        }
        your_schema = aggregate_to_schema(your_dict, REDDIT_MAP)
        zs_schema = run_zeroshot(text)

        all_labels = set(your_schema) | set(zs_schema)
        blended = {
            lbl: (0.40 * your_schema.get(lbl, 0.0) + 0.60 * zs_schema.get(lbl, 0.0))
            for lbl in all_labels
        }

        text_words = set(text.lower().split())
        if not text_words & DISTRESS_WORDS:
            blended["normal"] = blended.get("normal", 0) * 2.5
            total = sum(blended.values())
            blended = {k: v / total for k, v in blended.items()}

        zs_crisis = zs_schema.get("crisis", 0)
        your_crisis = your_schema.get("crisis", 0)
        if zs_crisis > 0.20 or your_crisis > 0.25:
            blended["crisis"] = max(blended.get("crisis", 0), zs_crisis, your_crisis)

        final_label = max(blended, key=blended.get)
        final_conf = blended[final_label]
        your_top = max(your_schema, key=your_schema.get)
        zs_top = max(zs_schema, key=zs_schema.get)
        agree = your_top == zs_top == final_label

        if final_label == "crisis":
            risk_level, status = "high", "condition_detected"
        elif final_conf >= threshold:
            risk_level = "high" if agree else "borderline"
            status = "condition_detected" if agree else "borderline"
        elif final_conf >= BORDERLINE_LOWER:
            risk_level, status = "borderline", "borderline"
        else:
            final_label = "normal"
            risk_level, status = "low", "normal"

        top5 = sorted(blended.items(), key=lambda x: -x[1])[:5]

        rows.append(
            {
                "participant_id": df["participant_id"].iloc[i],
                "classification": final_label,
                "confidence": round(final_conf, 4),
                "status": status,
                "risk_level": risk_level,
                "models_agree": agree,
                "your_model_top": your_top,
                "zeroshot_top": zs_top,
                "top1_condition": top5[0][0],
                "top1_prob": round(top5[0][1], 4),
                "top2_condition": top5[1][0] if len(top5) > 1 else "",
                "top2_prob": round(top5[1][1], 4) if len(top5) > 1 else 0.0,
                "word_count": df["word_count"].iloc[i],
            }
        )

        blended_full = {l: blended.get(l, 0.0) for l in SCHEMA_LABELS}
        blended_full["participant_id"] = df["participant_id"].iloc[i]
        blended_rows.append(blended_full)

        if (i + 1) % 5 == 0:
            print(f"   ... {i+1}/{len(texts)} done")

    global blended_matrix_df
    blended_matrix_df = pd.DataFrame(blended_rows).set_index("participant_id")

    return pd.DataFrame(rows)


print("\n✅ Ensemble functions defined — run next cell to classify")


In [ ]:
print("🚀 Running Ensemble Classification...")
print("=" * 70)

texts_list = df["text"].tolist()
n = len(texts_list)

results_df = classify_batch(texts_list)


n_normal = (results_df["risk_level"] == "low").sum()
n_borderline = (results_df["risk_level"] == "borderline").sum()
n_condition = (results_df["risk_level"] == "high").sum()
n_agree = results_df["models_agree"].sum()

print(f"\n✅ Done — {n} participants classified")
print(f"   🟢 Normal            : {n_normal}  ({n_normal/n*100:.1f}%)")
print(f"   🟡 Borderline        : {n_borderline}  ({n_borderline/n*100:.1f}%)")
print(f"   🔴 Condition detected: {n_condition}  ({n_condition/n*100:.1f}%)")
print(f"   🤝 Both models agreed: {n_agree}  ({n_agree/n*100:.1f}%)")
print("\n📊 Conditions found:")
cond_counts = results_df["classification"].value_counts()
max_cnt = cond_counts.max()
for cond, cnt in cond_counts.items():
    bar = "█" * int(cnt / max_cnt * 20)
    print(f"   {cond:<20} : {cnt:>3} ({cnt/n*100:.1f}%)  {bar}")


In [ ]:
print("📊 Generating Trend Analysis Dashboard...")

fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    f"Mental Health Trend Analysis  —  N={n} Participants",
    fontsize=17,
    fontweight="bold",
    y=0.98,
)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)


ax1 = fig.add_subplot(gs[0, 0])
cond_counts_plot = results_df["classification"].value_counts()
wedges, texts_p, autotexts = ax1.pie(
    cond_counts_plot.values,
    labels=cond_counts_plot.index,
    autopct="%1.1f%%",
    colors=PALETTE[: len(cond_counts_plot)],
    startangle=140,
    textprops={"fontsize": 8},
    pctdistance=0.8,
)
for at in autotexts:
    at.set_fontsize(7)
ax1.set_title("Condition Distribution", fontweight="bold", fontsize=12)


ax2 = fig.add_subplot(gs[0, 1])
risk_counts = results_df["risk_level"].value_counts()
risk_order = ["low", "borderline", "high"]
risk_labels = {"low": "Normal", "borderline": "Borderline", "high": "Detected"}
risk_colors = {"low": "#2ecc71", "borderline": "#f39c12", "high": "#e74c3c"}
present = [r for r in risk_order if r in risk_counts.index]
vals = [risk_counts[r] for r in present]
lbls = [risk_labels[r] for r in present]
cols = [risk_colors[r] for r in present]
bars = ax2.bar(lbls, vals, color=cols, edgecolor="white", linewidth=1.5, width=0.5)
for bar, val in zip(bars, vals):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f"{val}\n({val/n*100:.0f}%)",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )
ax2.set_ylabel("Participants", fontsize=10)
ax2.set_title("Mental Health Risk Level", fontweight="bold", fontsize=12)
ax2.set_ylim(0, max(vals) * 1.35)


ax3 = fig.add_subplot(gs[0, 2])
top_conds = cond_counts_plot.head(8)
colors_bar = [PALETTE[i] for i in range(len(top_conds))]
bars3 = ax3.barh(
    top_conds.index[::-1],
    top_conds.values[::-1],
    color=colors_bar[::-1],
    edgecolor="white",
)
for bar, val in zip(bars3, top_conds.values[::-1]):
    ax3.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f"{val}",
        va="center",
        fontsize=9,
        fontweight="bold",
    )
ax3.set_xlabel("Count", fontsize=10)
ax3.set_title("Top Detected Conditions", fontweight="bold", fontsize=12)
ax3.set_xlim(0, top_conds.max() * 1.3)


ax4 = fig.add_subplot(gs[1, 0])
conf_vals = results_df["confidence"]
ax4.hist(conf_vals, bins=15, color="#3498db", alpha=0.75, edgecolor="white")
ax4.axvspan(0, BORDERLINE_LOWER, alpha=0.08, color="#2ecc71", label="Normal zone")
ax4.axvspan(
    BORDERLINE_LOWER,
    CONFIDENCE_THRESHOLD,
    alpha=0.08,
    color="#f39c12",
    label="Borderline zone",
)
ax4.axvspan(
    CONFIDENCE_THRESHOLD, 1, alpha=0.08, color="#e74c3c", label="Condition zone"
)
ax4.axvline(
    conf_vals.mean(),
    color="#e74c3c",
    linestyle="--",
    lw=2,
    label=f"Mean={conf_vals.mean():.2f}",
)
ax4.axvline(
    conf_vals.median(),
    color="#9b59b6",
    linestyle="--",
    lw=2,
    label=f"Median={conf_vals.median():.2f}",
)
ax4.set_xlabel("Confidence Score", fontsize=10)
ax4.set_ylabel("Frequency", fontsize=10)
ax4.set_title("Confidence Distribution", fontweight="bold", fontsize=12)
ax4.legend(fontsize=7)


ax5 = fig.add_subplot(gs[1, 1:])
conditions_present = results_df["classification"].unique().tolist()
box_data = [
    results_df[results_df["classification"] == c]["confidence"].values
    for c in conditions_present
]
bp = ax5.boxplot(
    box_data, patch_artist=True, medianprops=dict(color="black", linewidth=2)
)
for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax5.set_xticks(range(1, len(conditions_present) + 1))
ax5.set_xticklabels(conditions_present, rotation=35, ha="right", fontsize=8)
ax5.set_ylabel("Confidence Score", fontsize=10)
ax5.set_title("Confidence by Condition", fontweight="bold", fontsize=12)
ax5.axhline(
    CONFIDENCE_THRESHOLD,
    color="#e74c3c",
    linestyle="--",
    lw=1.5,
    label=f"Condition threshold {CONFIDENCE_THRESHOLD}",
)
ax5.axhline(
    BORDERLINE_LOWER,
    color="#f39c12",
    linestyle="--",
    lw=1.5,
    label=f"Borderline threshold {BORDERLINE_LOWER}",
)
ax5.legend(fontsize=8)


ax6 = fig.add_subplot(gs[2, :])
heat_matrix = blended_matrix_df.values.T
im = ax6.imshow(heat_matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=0.7)
risk_icon = {"low": "N", "borderline": "B", "high": "!"}
for xi, pid in enumerate(blended_matrix_df.index):
    icon = risk_icon[
        results_df.loc[results_df["participant_id"] == pid, "risk_level"].values[0]
    ]
    ax6.text(xi, -0.8, icon, ha="center", va="top", fontsize=7, fontweight="bold")
ax6.set_xticks(range(n))
ax6.set_xticklabels(blended_matrix_df.index, rotation=90, fontsize=7)
ax6.set_yticks(range(len(blended_matrix_df.columns)))
ax6.set_yticklabels(blended_matrix_df.columns, fontsize=9)
ax6.set_title(
    "Blended Probability Heatmap  (N=Normal, B=Borderline, !=Detected)",
    fontweight="bold",
    fontsize=12,
)
plt.colorbar(im, ax=ax6, label="Blended Probability", fraction=0.02)

plt.savefig(
    _paths.output_path("trend_analysis_dashboard.png"), dpi=150, bbox_inches="tight"
)
plt.show()
print("\n✅ Dashboard saved → trend_analysis_dashboard.png")


In [ ]:
print("📊 Response Pattern Analysis...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Response Pattern Analysis", fontsize=14, fontweight="bold")


ax = axes[0]
ax.hist(
    results_df["word_count"], bins=12, color="#9b59b6", alpha=0.75, edgecolor="white"
)
ax.axvline(
    results_df["word_count"].mean(),
    color="#e74c3c",
    lw=2,
    linestyle="--",
    label=f"Mean={results_df['word_count'].mean():.0f} words",
)
ax.set_xlabel("Word Count per Response", fontsize=10)
ax.set_ylabel("Participants", fontsize=10)
ax.set_title("Response Length Distribution", fontweight="bold")
ax.legend(fontsize=9)


ax = axes[1]
merged = results_df[["classification", "word_count"]]
order = (
    merged.groupby("classification")["word_count"]
    .mean()
    .sort_values(ascending=False)
    .index
)
sns.boxplot(
    data=merged,
    x="classification",
    y="word_count",
    order=order,
    palette=PALETTE[: len(order)],
    ax=ax,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=8)
ax.set_xlabel("", fontsize=9)
ax.set_ylabel("Word Count", fontsize=10)
ax.set_title("Word Count by Condition", fontweight="bold")


ax = axes[2]
agree_counts = results_df["models_agree"].value_counts()
agree_labels = {True: "Both agree", False: "Models differ"}
ax.pie(
    agree_counts.values,
    labels=[agree_labels.get(k, str(k)) for k in agree_counts.index],
    autopct="%1.0f%%",
    colors=["#2ecc71", "#e74c3c"][: len(agree_counts)],
    startangle=90,
    textprops={"fontsize": 10},
)
ax.set_title("Model Agreement\n(Your model vs BART)", fontweight="bold")

plt.tight_layout()
plt.savefig(
    _paths.output_path("response_pattern_analysis.png"), dpi=150, bbox_inches="tight"
)
plt.show()
print("✅ Saved → response_pattern_analysis.png")


In [ ]:
print("🔬 SHAP XAI Analysis — Global Feature Importance (Your LR Model)")
print("=" * 70)

X_shap = tfidf.transform(texts_list)
feature_names = np.array(tfidf.get_feature_names_out())

print("⚙️  Computing SHAP values...")
shap_explainer = shap.LinearExplainer(lr, X_shap, feature_perturbation="interventional")
shap_vals = shap_explainer.shap_values(X_shap)
print(
    f"   ✅ SHAP computed for {len(shap_vals)} classes × {X_shap.shape[0]} participants"
)


detected_schema = (
    results_df[results_df["status"] == "condition_detected"]["classification"]
    .unique()
    .tolist()
)


def schema_to_idx(schema_label):
    if schema_label in list(le.classes_):
        return list(le.classes_).index(schema_label)
    return None


condition_indices = [
    i for i in [schema_to_idx(s) for s in detected_schema] if i is not None
]


if not condition_indices:
    condition_indices = list(range(len(le.classes_)))


mean_abs_shap = np.mean(
    [np.abs(shap_vals[i]).mean(axis=0) for i in condition_indices], axis=0
)

top20_idx = np.argsort(mean_abs_shap)[::-1][:20]
top20_words = feature_names[top20_idx]
top20_vals = mean_abs_shap[top20_idx]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    "SHAP — Global XAI: Most Influential Words (TF-IDF + LR Model)",
    fontsize=14,
    fontweight="bold",
)


ax = axes[0]
colors_shap = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 20))
ax.barh(top20_words[::-1], top20_vals[::-1], color=colors_shap[::-1])
ax.set_xlabel("Mean |SHAP Value|  (higher = more influential)", fontsize=10)
ax.set_title(
    "Top 20 Most Influential Words\n(across detected conditions)",
    fontweight="bold",
    fontsize=11,
)
ax.tick_params(axis="y", labelsize=9)


ax = axes[1]
top_classes = detected_schema[:5] if detected_schema else [le.classes_[0]]
plot_data = {}

for cname in top_classes:
    cidx = schema_to_idx(cname)
    if cidx is None:
        continue
    sv = shap_vals[cidx]
    mean_abs = np.abs(sv).mean(axis=0)
    top3_idx = np.argsort(mean_abs)[::-1][:3]

    for fi in top3_idx:
        plot_data[f"{feature_names[fi]}\n({cname})"] = mean_abs[fi]

if plot_data:
    words_pd = list(plot_data.keys())
    vals_pd = list(plot_data.values())

    sorted_pairs = sorted(zip(vals_pd, words_pd), reverse=True)
    vals_pd, words_pd = zip(*sorted_pairs)

    c_colors = [
        PALETTE[
            (
                top_classes.index(w.split("\n(")[1].rstrip(")"))
                if w.split("\n(")[1].rstrip(")") in top_classes
                else 0
            )
        ]
        for w in words_pd
    ]

    ax.barh(words_pd, vals_pd, color=c_colors, alpha=0.85, edgecolor="white")
    ax.set_xlabel("Mean |SHAP Value|", fontsize=10)
    ax.set_title(
        "Top 3 Key Words per Detected Condition", fontweight="bold", fontsize=11
    )

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=PALETTE[i]) for i in range(len(top_classes))
    ]
    ax.legend(handles, top_classes, fontsize=8)
    ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig(_paths.output_path("shap_global_xai.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ Saved → shap_global_xai.png")


In [ ]:
print("🔬 SHAP Deep Dive — Per Detected Condition")
print("=" * 70)

detected_conditions = (
    results_df[results_df["status"] == "condition_detected"]["classification"]
    .value_counts()
    .head(4)
    .index.tolist()
)
if not detected_conditions:
    print("⚠️  No conditions detected — showing top-2 schema labels")
    detected_conditions = (
        results_df["top1_condition"].value_counts().head(2).index.tolist()
    )

n_plots = len(detected_conditions)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 7))
if n_plots == 1:
    axes = [axes]
fig.suptitle(
    "SHAP Per-Condition: What Words Triggered Each Classification",
    fontsize=13,
    fontweight="bold",
)

for ax, cname in zip(axes, detected_conditions):
    cidx = schema_to_reddit_idx(cname)
    if cidx is None:
        ax.set_title(f"{cname} (no Reddit class)")
        continue

    sv = shap_vals[cidx]
    pos_mean = np.where(sv > 0, sv, 0).mean(axis=0)
    neg_mean = np.where(sv < 0, sv, 0).mean(axis=0)

    top_pos_idx = np.argsort(pos_mean)[::-1][:8]
    top_neg_idx = np.argsort(neg_mean)[:8]
    combined_idx = list(dict.fromkeys(list(top_pos_idx) + list(top_neg_idx)))[:12]

    words_plot = feature_names[combined_idx]
    vals_plot = sv.mean(axis=0)[combined_idx]
    sorted_pairs = sorted(zip(vals_plot, words_plot))
    vals_plot, words_plot = zip(*sorted_pairs)

    bar_colors = ["#e74c3c" if v > 0 else "#3498db" for v in vals_plot]
    ax.barh(words_plot, vals_plot, color=bar_colors, alpha=0.85, edgecolor="white")
    ax.axvline(0, color="black", lw=1)
    ax.set_title(f"[{cname.upper()}]", fontweight="bold", fontsize=11)
    ax.set_xlabel("Mean SHAP Value", fontsize=9)
    ax.tick_params(axis="y", labelsize=9)
    from matplotlib.patches import Patch

    ax.legend(
        handles=[
            Patch(color="#e74c3c", label="pushes TOWARD"),
            Patch(color="#3498db", label="pushes AWAY"),
        ],
        fontsize=8,
        loc="lower right",
    )

plt.tight_layout()
plt.savefig(_paths.output_path("shap_per_condition.png"), dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Saved → shap_per_condition.png")

print("\n📋 SEMANTIC SUMMARY")
print("=" * 70)
for cname in detected_conditions:
    cidx = schema_to_reddit_idx(cname)
    if cidx is None:
        continue
    sv = shap_vals[cidx]
    pos_mean = np.where(sv > 0, sv, 0).mean(axis=0)
    top_words = feature_names[np.argsort(pos_mean)[::-1][:8]]
    n_in_cond = (results_df["classification"] == cname).sum()
    print(f"\n  [{cname.upper()}]  ({n_in_cond} participant(s))")
    print(f"     Key words: {', '.join(top_words)}")


In [ ]:
print("👤 Individual Participant Results")
print("=" * 70)

display_cols = [
    "participant_id",
    "classification",
    "confidence",
    "status",
    "top1_condition",
    "top1_prob",
    "top2_condition",
    "top2_prob",
    "models_agree",
    "your_model_top",
    "zeroshot_top",
    "word_count",
]
display_df = results_df[display_cols].copy()
display_df["confidence"] = display_df["confidence"].apply(lambda x: f"{x:.1%}")
display_df["top1_prob"] = display_df["top1_prob"].apply(lambda x: f"{x:.1%}")
display_df["top2_prob"] = display_df["top2_prob"].apply(lambda x: f"{x:.1%}")

print(
    f"{'ID':<8} {'Predicted':<20} {'Conf':<8} {'YourModel':<18} {'BART':<18} {'Agree':<7} {'Status'}"
)
print("-" * 90)
for _, row in results_df.iterrows():
    icon = (
        "N"
        if row["status"] == "normal"
        else ("B" if row["status"] == "borderline" else "!")
    )
    print(
        f"{row['participant_id']:<8} {row['classification']:<20} {row['confidence']:.0%}    "
        f"{row['your_model_top']:<18} {row['zeroshot_top']:<18} {str(row['models_agree']):<7} [{icon}] {row['status']}"
    )

try:
    from IPython.display import display as ipy_display

    def status_color(val):
        if val == "normal":
            return "background-color:#d5f5e3"
        if val == "borderline":
            return "background-color:#fef9e7"
        return "background-color:#fadbd8"

    styled = (
        display_df.style.applymap(status_color, subset=["status"])
        .set_properties(**{'font-size': '11px'})
        .set_caption(f"Ensemble Classification — {n} Participants")
    )
    ipy_display(styled)
except:
    print(display_df.to_string())


In [ ]:
print("💾 Exporting results...")
print("=" * 70)

results_df.to_csv(_paths.output_path("mental_health_results_full.csv"), index=False)
print("   ✅ mental_health_results_full.csv")

cond_dist = results_df["classification"].value_counts().reset_index()
cond_dist.columns = ["condition", "count"]
cond_dist["percentage"] = (cond_dist["count"] / n * 100).round(1)
cond_dist.to_csv(_paths.output_path("mental_health_distribution.csv"), index=False)
print("   ✅ mental_health_distribution.csv")

blended_matrix_df.to_csv(_paths.output_path("mental_health_blended_proba.csv"))
print("   ✅ mental_health_blended_proba.csv  (full probability matrix)")

summary = {
    "total_participants": [n],
    "normal_count": [n_normal],
    "borderline_count": [n_borderline],
    "condition_detected_count": [n_condition],
    "normal_pct": [f"{n_normal/n*100:.1f}%"],
    "condition_pct": [f"{n_condition/n*100:.1f}%"],
    "mean_confidence": [f"{results_df['confidence'].mean():.1%}"],
    "median_confidence": [f"{results_df['confidence'].median():.1%}"],
    "models_agreed_pct": [f"{n_agree/n*100:.1f}%"],
    "threshold_used": [CONFIDENCE_THRESHOLD],
    "ensemble": ["Reddit TF-IDF (LR+SVM+RF) 40% + BART zero-shot 60%"],
}
pd.DataFrame(summary).T.reset_index().rename(
    columns={"index": "metric", 0: "value"}
).to_csv(_paths.output_path("mental_health_summary.csv"), index=False)
print("   ✅ mental_health_summary.csv")

merged_out = pd.concat(
    [raw_df.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1
)
merged_out.to_csv(_paths.output_path("mental_health_merged.csv"), index=False)
print("   ✅ mental_health_merged.csv")

print("\n✅ All exports complete!")
print("\n⚠️  Research disclaimer:")
print("   This is a screening tool, NOT a clinical diagnosis.")
print("   Always involve qualified mental health professionals.")


## 📄 Research Paper — Methods Snippet

Copy into your paper's **Methods** section:

> *Text responses were analysed using a two-model ensemble. First, a Reddit-trained TF-IDF ensemble (Logistic Regression, Linear SVM, Random Forest; trained on 457,255 posts across 15 mental-health subreddits) provided clinical-vocabulary signal with class-prior bias correction. Second, a zero-shot NLI classifier (facebook/bart-large-mnli; Lewis et al., 2020) provided domain-agnostic semantic classification using seven survey-tailored candidate labels. Final labels were determined by a weighted blend (40% TF-IDF ensemble, 60% zero-shot), with a crisis-escalation override and a normal-class boost for responses lacking distress vocabulary. Model agreement was logged as an inter-model reliability metric. SHAP LinearExplainer (Lundberg & Lee, 2017) was applied to the Logistic Regression component to identify influential linguistic features. All analysis was performed in Python 3.10.*

**Cite:**
- Lewis et al. (2020). BART: Denoising Sequence-to-Sequence Pre-training. *ACL 2020*.
- Lundberg & Lee (2017). A unified approach to interpreting model predictions. *NeurIPS 2017*.